 # **Lab 04 – Spark Streaming**

## **Group: CDHD**

### **Team Members**

| STT | Full Name           | Student ID |
| :-: | ------------------- | :--------: |
|  1  | Phan Thị Phương Chi |  23120025  |
|  2  | Trần Thanh Đạt      |  23120030  |
|  3  | Phạm Ngọc Duy       |  23120035  |
|  4  | Lê Hoàng Mỹ Hạ      |  23120038  |

## **Task 1: Repository Cloning and File Discovery**

### Mục tiêu
- Khám phá toàn bộ kho mã nguồn Python được chọn làm **corpus** cho CPG Parser Service (Task 2).
- Kết quả của task này là danh sách file `.py` đã phân loại, lưu vào `output/discovered_files.csv`. 
  
### Thông tin repo

| Trường        | Giá trị |
|:--------------|:--------|
| **Tên repo**  | `peft` |
| **Tổ chức**   | `huggingface` |
| **URL clone** | `https://github.com/huggingface/peft.git` |

### 0. Imports & cấu hình đường dẫn

In [1]:
import os
import sys
import subprocess
import platform
from pathlib import Path

import pandas as pd

# Thư mục gốc của project (thư mục chứa notebook)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "peft").exists() and (PROJECT_ROOT.parent / "peft").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Đường dẫn repository và thư mục output trên Windows
WIN_REPO_PATH = str(PROJECT_ROOT / "peft")
WIN_OUTPUT_DIR = str(PROJECT_ROOT / "output")

# Đường dẫn tương ứng khi chạy trong WSL
# Ví dụ:
# C:\Users\...\spark-streaming-lab
# -> /mnt/c/Users/.../spark-streaming-lab
WSL_REPO_PATH = "/mnt/c/Users/biend/OneDrive/Máy tính/spark-streaming-lab/peft"
WSL_OUTPUT_DIR = "/mnt/c/Users/biend/OneDrive/Máy tính/spark-streaming-lab/output"

REPO_CLONE_URL = "https://github.com/huggingface/peft.git"

# Chọn đường dẫn phù hợp với hệ điều hành hiện tại
if platform.system() == "Windows":
    REPO_ROOT = Path(WIN_REPO_PATH)
    OUTPUT_DIR = Path(WIN_OUTPUT_DIR)
else:
    REPO_ROOT = Path(WSL_REPO_PATH)
    OUTPUT_DIR = Path(WSL_OUTPUT_DIR)

# Tạo thư mục output nếu chưa tồn tại
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python     : {sys.version}")
print(f"Hệ điều hành: {platform.system()} - {platform.version()}")
print(f"PROJECT    : {PROJECT_ROOT}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# Thêm thư mục project vào PYTHONPATH để có thể import task1, task4
sys.path.insert(
    0,
    str(Path(os.path.abspath(".")).parent)
    if Path(os.path.abspath(".")).name == "task1"
    else str(Path(os.path.abspath(".")))
)

from task1.clone import clone_repo_if_needed
from task1.discover import (
    is_auto_generated,
    classify_file,
    count_lines,
    scan_repo,
    build_dataframe,
    smoke_test,
)

Python     : 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Hệ điều hành: Windows - 10.0.26200
PROJECT    : E:\BD
REPO_ROOT  : E:\BD\peft
OUTPUT_DIR : E:\BD\output


### 1. Kiểm tra & Clone Repository

In [2]:
clone_repo_if_needed(REPO_ROOT, REPO_CLONE_URL)

assert REPO_ROOT.exists(), f"[LỖI] Không tìm thấy repository tại {REPO_ROOT}"
print(f"[OK] Repository hợp lệ: {REPO_ROOT}")

[INFO] Repository đã tồn tại tại: E:\BD\peft
[INFO] Bỏ qua bước clone.
[OK] Repository hợp lệ: E:\BD\peft


### 2. Duyệt đệ quy & thu thập tất cả file `.py`

In [3]:
all_py_files_relative, categories = scan_repo(REPO_ROOT)

print("\nVí dụ 10 tệp đầu tiên:")
for p in all_py_files_relative[:10]:
    print(f"  {p}")

[OK] Tìm thấy 431 tệp .py

Ví dụ 10 tệp đầu tiên:
  docs\source\_config.py
  examples\adamss_finetuning\glue_adamss_asa_example.py
  examples\adamss_finetuning\glue_adamss_asa_manual_example.py
  examples\adamss_finetuning\image_classification_adamss_asa.py
  examples\adamss_finetuning\test_adamss_quick.py
  examples\alora_finetuning\alora_finetuning.py
  examples\arrow_multitask\arrow_phi3_mini.py
  examples\bdlora_finetuning\chat.py
  examples\beft_finetuning\beft_finetuning.py
  examples\boft_controlnet\__init__.py


### 3. Phân loại file `.py`

**Mục tiêu:** Loại bỏ các file không liên quan trước khi đưa vào CPG Parser, giúp giảm nhiễu
và tăng chất lượng đồ thị phụ thuộc.

**Bốn nhóm phân loại và lý do loại trừ:**

| Nhóm | Tiêu chí | Lý do loại trừ |
|:-----|:---------|:---------------|
| **test** | Thư mục chứa `test`/`tests`; tên file bắt đầu bằng `test_` | Test code không đại diện cho logic nghiệp vụ, chứa mock và fixture gây nhiễu CPG |
| **setup** | Tên file chính xác là `setup.py`, `__init__.py`, `conftest.py` (ngoài thư mục test) | File scaffolding/packaging, không chứa logic domain |
| **auto-generated** | Nằm trong `build/`, `dist/`, `*.egg-info/`; hoặc header chứa `DO NOT EDIT` / `auto-generated` | Code sinh tự động thường bị flatten, không phản ánh cấu trúc thiết kế thực |
| **source** | Tất cả file còn lại | Corpus chính cho CPG Parser Service ở Task 2 |

**Thứ tự ưu tiên phân loại:** `auto-generated` -> `setup` (exact filename) -> `test` (path/prefix) -> `source`

In [4]:
from collections import Counter

category_counts = Counter(categories)

print("=" * 50)
print("KẾT QUẢ PHÂN LOẠI TỆP .py")
print("=" * 50)

for cat in ["source", "test", "setup", "auto-generated"]:
    print(f"  {cat:<20}: {category_counts.get(cat, 0):>5} tệp")

print("-" * 50)
print(f'  {"TỔNG":<20}: {sum(category_counts.values()):>5} tệp')

# Kiểm tra conftest.py phải được phân loại là setup
conftest = [
    (p, c)
    for p, c in zip(all_py_files_relative, categories)
    if p.name == "conftest.py"
]

print("\nKiểm tra conftest.py (phải là setup):")

for path, cat in conftest[:5]:
    status = "OK" if cat == "setup" else "LỖI"
    print(f"  [{status}] {path} -> {cat}")

KẾT QUẢ PHÂN LOẠI TỆP .py
  source              :   309 tệp
  test                :    63 tệp
  setup               :    59 tệp
  auto-generated      :     0 tệp
--------------------------------------------------
  TỔNG                :   431 tệp

Kiểm tra conftest.py (phải là setup):
  [OK] tests\conftest.py -> setup


**Kiểm chứng heuristic is_auto_generated() (Smoke Test)**  
Do repository ban đầu không chứa các file sinh tự động nên kết quả phát hiện là 0 file. Để kiểm tra hàm `is_auto_generated()`, nhóm tạo một số file và thư mục giả lập (ví dụ build/, dist/,...) rồi chạy lại chương trình. Nếu các file này được phát hiện đúng là auto-generated thì có thể kết luận detector hoạt động như mong đợi. Sau khi kiểm tra, nhóm xóa các file giả lập để giữ nguyên trạng thái của repository.

In [5]:
smoke_test()

  [OK] Tệp có 'DO NOT EDIT'
  [OK] Tệp Python thông thường
  [OK] Đường dẫn chứa 'build/'
[OK] Smoke test thành công (3/3)


### 4. Tạo DataFrame & Lưu ra CSV

In [6]:
df = build_dataframe(REPO_ROOT, all_py_files_relative, categories, OUTPUT_DIR)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

print("\n10 dòng đầu:")
df.head(10)

[OK] Đã lưu tệp CSV: E:\BD\output\discovered_files.csv (431 dòng)

10 dòng đầu:


,relative_path,size_bytes,category,num_lines
0,docs/source/_config.py,287,source,7
1,examples/adamss_finetuning/glue_adamss_asa_example.py,15224,source,382
2,examples/adamss_finetuning/glue_adamss_asa_manual_example.py,16825,source,411
3,examples/adamss_finetuning/image_classification_adamss_asa.py,17368,source,452
4,examples/adamss_finetuning/test_adamss_quick.py,4587,test,176
5,examples/alora_finetuning/alora_finetuning.py,9839,source,259
6,examples/arrow_multitask/arrow_phi3_mini.py,14755,source,383
7,examples/bdlora_finetuning/chat.py,1972,source,64
8,examples/beft_finetuning/beft_finetuning.py,4083,source,122
9,examples/boft_controlnet/__init__.py,0,setup,0


In [7]:
summary_df = (
    df.groupby("category")
      .agg(
          num_files=("relative_path", "count"),
          total_bytes=("size_bytes", "sum"),
          total_lines=("num_lines", "sum"),
          avg_lines=("num_lines", "mean"),
      )
      .reset_index()
      .sort_values("num_files", ascending=False)
)

summary_df["avg_lines"] = summary_df["avg_lines"].round(1)

print("Thống kê theo từng loại tệp:")
summary_df

Thống kê theo từng loại tệp:


,category,num_files,total_bytes,total_lines,avg_lines
1,source,309,4052222,93524,302.7
2,test,63,2065517,48096,763.4
0,setup,59,75268,2144,36.3


### 5. Bảng tổng kết cuối cùng (Lab Report)

In [8]:
total_py_files = len(df)
n_test         = category_counts.get('test', 0)
n_setup        = category_counts.get('setup', 0)
n_autogen      = category_counts.get('auto-generated', 0)
n_excluded     = n_test + n_setup + n_autogen
n_source       = category_counts.get('source', 0)

print('╔══════════════════════════════════════════════════════╗')
print('║           TASK 1 – FINAL SUMMARY (Lab Report)        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Repository : huggingface/peft                       ║')
print(f'║  URL        : https://github.com/huggingface/peft    ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Total .py files discovered        : {total_py_files:>5}           ║')
print('╠──────────────────────────────────────────────────────╣')
print(f'║  Excluded – test files             : {n_test:>5}           ║')
print(f'║  Excluded – setup/init files       : {n_setup:>5}           ║')
print(f'║  Excluded – auto-generated files   : {n_autogen:>5}           ║')
print(f'║  Total excluded                    : {n_excluded:>5}           ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  SOURCE files -> CPG Parser Service : {n_source:>5}          ║')
print('╚══════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════╗
║           TASK 1 – FINAL SUMMARY (Lab Report)        ║
╠══════════════════════════════════════════════════════╣
║  Repository : huggingface/peft                       ║
║  URL        : https://github.com/huggingface/peft    ║
╠══════════════════════════════════════════════════════╣
║  Total .py files discovered        :   431           ║
╠──────────────────────────────────────────────────────╣
║  Excluded – test files             :    63           ║
║  Excluded – setup/init files       :    59           ║
║  Excluded – auto-generated files   :     0           ║
║  Total excluded                    :   122           ║
╠══════════════════════════════════════════════════════╣
║  SOURCE files -> CPG Parser Service :   309          ║
╚══════════════════════════════════════════════════════╝


### 6. Reflection

**What worked**
- `git clone --depth 1` giúp lấy repo `peft` nhanh và nhẹ, đủ dùng cho việc phân tích tĩnh mã nguồn.
- `pathlib.Path.rglob("*.py")` duyệt đệ quy toàn bộ 431 file `.py` mà không cần xử lý riêng cho từng hệ điều hành.
- Heuristic phân loại 4 nhóm (`auto-generated` -> `setup` -> `test` -> `source`) cho kết quả nhất quán, đã kiểm chứng bằng smoke test riêng cho `is_auto_generated()` (3/3 test case PASS).

**What failed**
- Bản đầu tiên đặt thứ tự ưu tiên là `test` trước `setup`, khiến `conftest.py` bị nhận nhầm thành `test` (do `"test"` là substring của `"conftest"`). Notebook đã bổ sung một bước kiểm tra riêng (in ra danh sách `conftest.py` và category tương ứng) để phát hiện lỗi này.
- Repo `peft` không có sẵn file/thư mục auto-generated (`build/`, `dist/`, `.egg-info/`) nên nhóm không thể xác nhận `is_auto_generated()` hoạt động đúng chỉ bằng cách chạy trên repo thật.

**How resolved**
- Đổi thứ tự ưu tiên phân loại: kiểm tra khớp tên file chính xác (`setup`) trước khi kiểm tra từ khoá `test`, giải quyết dứt điểm lỗi `conftest.py`.
- Viết smoke test tạo file/thư mục giả lập (có marker `DO NOT EDIT`, path chứa `build/`) để kiểm chứng `is_auto_generated()` hoạt động đúng, bù lại việc repo thật không có ca auto-generated nào.

## **Task 2: Incremental CPG Parser Service**

### Mục tiêu
- Xây dựng **Parser Service** (`cpg_parser.py`) xử lý file Python **từng file một** (không parse cả repo một lần).
- Dùng thư viện **`ast`** (Python standard library) để trích xuất:
  - **AST nodes**
  - **CFG edges** (control flow)
  - **DFG edges** (data flow)
  - **CALL edges** (function/method calls)
- Mỗi phần tử có **ID ổn định** (UUIDv5) để reprocess không tạo trùng downstream.
- Emit 3 luồng sự kiện chính: **nodes**, **edges**, **metadata** (+ `errors` khi parse lỗi).
- Service hoạt động với **bounded memory**: parse 1 file → emit → flush → sang file tiếp theo.

### Thiết kế nhanh

| Thành phần | Cách làm |
|:---|:---|
| Parser | `ast.parse` + `ast.NodeVisitor` trong `cpg_parser.py` |
| Stable node ID | `uuid.uuid5(NAMESPACE_DNS, f"{file_path}:{ast_path}")` |
| Stable edge ID | `uuid.uuid5(NAMESPACE_DNS, f"{src}->{tgt}:{edge_type}")` |
| Metadata ID | `uuid.uuid5(NAMESPACE_DNS, file_path)` |
| Input | `output/discovered_files.csv` (chỉ `category == source`) |
| Chế độ chạy | `--dry-run` (không Kafka) / emit Kafka (`--bootstrap-servers`) |


### 0. Kiểm tra đầu vào Task 1 & cấu hình Parser


In [ ]:
import json
import hashlib
import uuid
from collections import Counter

# Dùng lại PROJECT_ROOT / REPO_ROOT / OUTPUT_DIR từ cell cấu hình Task 1
assert "PROJECT_ROOT" in dir(), "Hãy chạy cell cấu hình đường dẫn (Task 1) trước."

PARSER_SCRIPT = PROJECT_ROOT / "cpg_parser.py"
DISCOVERED_CSV = OUTPUT_DIR / "discovered_files.csv"
KAFKA_BOOTSTRAP = "localhost:9092"
SCHEMA_VERSION = "1.0.0"
DEMO_LIMIT = 5  # số file dùng cho demo notebook (đủ bằng chứng, không quá lâu)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("REPO_ROOT      :", REPO_ROOT)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("PARSER_SCRIPT  :", PARSER_SCRIPT, "| exists =", PARSER_SCRIPT.exists())
print("DISCOVERED_CSV :", DISCOVERED_CSV, "| exists =", DISCOVERED_CSV.exists())
print("Kafka brokers  :", KAFKA_BOOTSTRAP)
print("Schema version :", SCHEMA_VERSION)
print("Demo limit     :", DEMO_LIMIT)

assert PARSER_SCRIPT.exists(), f"Thiếu {PARSER_SCRIPT}"
assert DISCOVERED_CSV.exists(), f"Thiếu {DISCOVERED_CSV} — chạy Task 1 trước"
assert REPO_ROOT.exists(), f"Thiếu repo tại {REPO_ROOT}"

df_all = pd.read_csv(DISCOVERED_CSV)
df_source = df_all[df_all["category"] == "source"].copy()
print(f"\n[OK] CSV có {len(df_all)} file .py, trong đó source = {len(df_source)}")
print(df_source.head(5).to_string(index=False))


### 1. Dry-run Parser Service (không gửi Kafka)

Chạy `cpg_parser.py --dry-run` trên vài file nguồn đầu tiên để lấy:
- số node/edge từng file
- **sample JSON** cho node / edge / metadata
- xác nhận mỗi message có `schema_version` và `event_time`


In [ ]:
#!pip install kafka-python

In [ ]:
cmd_dry = [
    sys.executable, str(PARSER_SCRIPT),
    "--dry-run",
    "--limit", str(DEMO_LIMIT),
    "--repo-root", str(REPO_ROOT),
    "--discovered-csv", str(DISCOVERED_CSV),
    "--schema-version", SCHEMA_VERSION,
]
#print("CMD:", " ".join(cmd_dry))
print("-" * 72)

proc = subprocess.run(
    cmd_dry,
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(proc.stdout)
if proc.returncode != 0:
    print("[STDERR]")
    print(proc.stderr)
    raise RuntimeError(f"Dry-run thất bại, exit={proc.returncode}")

log_dry = OUTPUT_DIR / "task2_dryrun.log"
log_dry.write_text(proc.stdout, encoding="utf-8")
print(f"[OK] Đã lưu log -> {log_dry}")


### 2. Phân tích chi tiết 1 file + kiểm chứng Stable ID (idempotency)

Parse cùng một file **hai lần**. Nếu ID ổn định, tập `id` của nodes/edges/metadata lần 1 và lần 2 phải **trùng khớp 100%**.


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from cpg_parser import process_file, normalize_rel_path

sample_rel = df_source.iloc[0]["relative_path"]
print("Sample file:", sample_rel)

nodes1, edges1, meta1 = process_file(sample_rel, REPO_ROOT, SCHEMA_VERSION)
nodes2, edges2, meta2 = process_file(sample_rel, REPO_ROOT, SCHEMA_VERSION)

ids_n1 = {n["id"] for n in nodes1}
ids_n2 = {n["id"] for n in nodes2}
ids_e1 = {e["id"] for e in edges1}
ids_e2 = {e["id"] for e in edges2}

print(f"Nodes  run1={len(nodes1)} run2={len(nodes2)} | ID overlap = {len(ids_n1 & ids_n2)}/{len(ids_n1)}")
print(f"Edges  run1={len(edges1)} run2={len(edges2)} | ID overlap = {len(ids_e1 & ids_e2)}/{len(ids_e1)}")
print(f"Metadata id run1={meta1['id']}")
print(f"Metadata id run2={meta2['id']}")
print(f"Metadata ID identical? {meta1['id'] == meta2['id']}")

assert ids_n1 == ids_n2, "Node IDs không ổn định giữa 2 lần parse!"
assert ids_e1 == ids_e2, "Edge IDs không ổn định giữa 2 lần parse!"
assert meta1["id"] == meta2["id"], "Metadata ID không ổn định!"
print("\n[OK] Stable ID verified — reprocess cùng nội dung không tạo ID mới.")

edge_types = Counter(e["type"] for e in edges1)
print("\nEdge type counts:", dict(edge_types))

print("\n--- SAMPLE NODE ---")
print(json.dumps(nodes1[0], indent=2, ensure_ascii=False))
print("\n--- SAMPLE EDGE ---")
print(json.dumps(edges1[0], indent=2, ensure_ascii=False))
print("\n--- SAMPLE METADATA ---")
print(json.dumps(meta1, indent=2, ensure_ascii=False))

for label, obj in [("node", nodes1[0]), ("edge", edges1[0]), ("metadata", meta1)]:
    assert "schema_version" in obj, f"{label} thiếu schema_version"
    assert "event_time" in obj, f"{label} thiếu event_time"
print("\n[OK] schema_version + event_time có trên mọi loại event.")


### 3. Tổng hợp số liệu dry-run trên DEMO_LIMIT file nguồn

Parse lần lượt `DEMO_LIMIT` file trong notebook để ghi **bảng số liệu** (bằng chứng Task 2).


In [ ]:
rows = []
total_nodes = total_edges = 0
edge_counter = Counter()

for rel in df_source["relative_path"].head(DEMO_LIMIT).tolist():
    nodes, edges, meta = process_file(rel, REPO_ROOT, SCHEMA_VERSION)
    et = Counter(e["type"] for e in edges)
    rows.append({
        "file_path": meta["file_path"],
        "num_lines": meta["num_lines"],
        "sha256_short": meta["sha256"][:12],
        "nodes": len(nodes),
        "edges": len(edges),
        "AST": et.get("AST", 0),
        "CFG": et.get("CFG", 0),
        "DFG": et.get("DFG", 0),
        "CALL": et.get("CALL", 0),
    })
    total_nodes += len(nodes)
    total_edges += len(edges)
    edge_counter.update(et)

stats_df = pd.DataFrame(rows)
display(stats_df)

print("=== TASK 2 DRY-RUN SUMMARY ===")
print(f"Files parsed : {len(rows)}")
print(f"Total nodes  : {total_nodes}")
print(f"Total edges  : {total_edges}")
print(f"Edge breakdown: {dict(edge_counter)}")

stats_path = OUTPUT_DIR / "task2_parse_stats.csv"
stats_df.to_csv(stats_path, index=False)
print(f"[OK] Saved stats -> {stats_path}")


### 4. Reflection – Task 2

**What worked**
- `ast` đủ để dựng AST parent–child, suy CFG tuần tự trên statement/expression, DFG định nghĩa→dùng, và CALL từ function bao quanh tới `ast.Call`.
- UUIDv5 + đường dẫn POSIX (`as_posix`) giúp ID ổn định giữa Windows/WSL.
- Xử lý từng file + `--limit` / `--file` giúp demo an toàn, tránh OOM.

**What failed / lưu ý**
- CFG/DFG ở đây mang tính demo giáo dục (không phải phân tích liên thủ tục đầy đủ như Joern).
- File syntax lỗi sẽ không tạo nodes — được ghi vào topic `cpg-errors` khi emit Kafka.

**How resolved**
- Chuẩn hoá `file_path` trước khi hash ID; flush Kafka sau mỗi file; thêm `--dry-run` để kiểm chứng trước khi ghi broker.


## **Task 3: Kafka Topic Design**

### Mục tiêu
- Thiết kế **4 Kafka topic** mang event từ Parser Service:
  1. `cpg-nodes` — node events
  2. `cpg-edges` — edge events (AST/CFG/DFG/CALL)
  3. `cpg-metadata` — source metadata events
  4. `cpg-errors` — parser error events
- Mỗi message bắt buộc có **`schema_version`** (forward compatibility) và **`event_time`**.
- Chứng minh Parser (Task 2) **đã ghi thành công** vào broker: list topic, offset/count, sample message.

### Sơ đồ luồng

```
Python source files (.py)
        |
        v
   cpg_parser.py  (incremental, 1 file at a time)
        |
        |---> cpg-nodes      ---> Neo4j Sink (Task 4)
        |---> cpg-edges      ---> Neo4j Sink (Task 4)
        |---> cpg-metadata   ---> Spark -> MongoDB (Task 5)
        +---> cpg-errors     ---> monitoring / debug
```


### 0. Schema thiết kế (message contract)

#### Topic `cpg-nodes`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable node id — **Kafka key** |
| `type` / `label` | string | Loại AST node (`FunctionDef`, `Name`, ...) |
| `file_path` | string | Đường dẫn tương đối (POSIX) |
| `start_line`, `start_column`, `end_line`, `end_column` | int/null | Vị trí nguồn |
| `code` | string | Snippet |
| `properties` | object | name/value phụ |
| `schema_version` | string | vd. `1.0.0` |
| `event_time` | string | ISO-8601 UTC |

#### Topic `cpg-edges`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable edge id — **Kafka key** |
| `source_id`, `target_id` | string | Liên kết tới node ids |
| `type` | string | `AST` / `CFG` / `DFG` / `CALL` |
| `properties` | object (optional) | vd. `callee` cho CALL |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-metadata`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable file id — **Kafka key** |
| `file_path` | string | Đường dẫn tương đối |
| `size_bytes`, `num_lines` | int | Kích thước file |
| `sha256` | string | Hash nội dung |
| `processed_at`, `status` | string | Thời điểm / SUCCESS |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-errors`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | `uuid5(file_path + ":error")` |
| `file_path`, `error_type`, `error_message`, `stack_trace` | string | Chi tiết lỗi |
| `occurred_at`, `schema_version`, `event_time` | string | Bắt buộc |

**Cấu hình broker (lab):** `replication-factor=1`, `partitions=1` (single-broker WSL).


### 1. Khởi động Kafka (WSL) & tạo 4 topic

Cell dưới gọi script `scripts/setup_kafka.sh` (start ZooKeeper + Broker + create topics nếu chưa có).


In [ ]:
SETUP_SCRIPT_WSL = "/mnt/c/Users/biend/OneDrive/Máy tính/spark-streaming-lab/scripts/setup_kafka.sh"
KAFKA_HOME = "/home/pnd/data/kafka_2.12-3.8.0"

def run_wsl(bash_cmd, timeout=180):
    """Chạy lệnh bash trong WSL, trả về CompletedProcess."""
    return subprocess.run(
        ["wsl", "-e", "bash", "-c", bash_cmd],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )

print("[INFO] Chạy setup_kafka.sh ...")
setup_cmd = "sed -i 's/\\r$//' '" + SETUP_SCRIPT_WSL + "' && bash '" + SETUP_SCRIPT_WSL + "'"
r = run_wsl(setup_cmd, timeout=180)
print(r.stdout)
if r.returncode != 0:
    print("[STDERR]", r.stderr)
    raise RuntimeError("setup_kafka.sh thất bại")

print("\n[INFO] kafka-topics --list")
r2 = run_wsl(KAFKA_HOME + "/bin/kafka-topics.sh --list --bootstrap-server localhost:9092")
print(r2.stdout)
topics_found = [t.strip() for t in r2.stdout.splitlines() if t.strip()]
required = {"cpg-nodes", "cpg-edges", "cpg-metadata", "cpg-errors"}
missing = required - set(topics_found)
print("Topics found:", topics_found)
assert not missing, f"Thiếu topic: {missing}"
print("[OK] Đủ 4 topic Task 3 trên broker.")

(OUTPUT_DIR / "task3_topics_list.log").write_text(r2.stdout, encoding="utf-8")


### 2. Describe topics (bằng chứng cấu hình partition / replication)


In [ ]:
r3 = run_wsl(KAFKA_HOME + "/bin/kafka-topics.sh --describe --bootstrap-server localhost:9092")
print(r3.stdout)
(OUTPUT_DIR / "task3_topics_describe.log").write_text(r3.stdout, encoding="utf-8")
print(f"[OK] Saved -> {OUTPUT_DIR / 'task3_topics_describe.log'}")


### 3. Emit sự kiện từ Parser Service lên Kafka (bằng chứng end-to-end)

Chạy `cpg_parser.py` **không** `--dry-run` với `--limit DEMO_LIMIT` để ghi thật vào 4 topic.


In [ ]:
cmd_emit = [
    sys.executable, str(PARSER_SCRIPT),
    "--limit", str(DEMO_LIMIT),
    "--repo-root", str(REPO_ROOT),
    "--discovered-csv", str(DISCOVERED_CSV),
    "--bootstrap-servers", KAFKA_BOOTSTRAP,
    "--schema-version", SCHEMA_VERSION,
]
print("CMD:", " ".join(cmd_emit))
print("-" * 72)

proc_emit = subprocess.run(
    cmd_emit,
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(proc_emit.stdout)
if proc_emit.returncode != 0:
    print("[STDERR]")
    print(proc_emit.stderr)
    raise RuntimeError(f"Emit Kafka thất bại, exit={proc_emit.returncode}")

log_emit = OUTPUT_DIR / "task3_emit.log"
log_emit.write_text(proc_emit.stdout, encoding="utf-8")
print(f"[OK] Đã emit và lưu log -> {log_emit}")


### 4. Đếm message trên từng topic (offset) — bằng chứng Parser đã ghi


In [ ]:
from kafka import KafkaConsumer, TopicPartition

topics = ["cpg-nodes", "cpg-edges", "cpg-metadata", "cpg-errors"]
consumer = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP, request_timeout_ms=15000)

print("=== Kafka topic offsets (bằng chứng ghi thành công) ===")
offset_rows = []
for t in topics:
    tp = TopicPartition(t, 0)
    begin = consumer.beginning_offsets([tp])[tp]
    end = consumer.end_offsets([tp])[tp]
    count = end - begin
    offset_rows.append({"topic": t, "begin": begin, "end": end, "messages": count})
    print(f"  {t:14s}  begin={begin:<8d} end={end:<8d} messages={count}")

consumer.close()

offset_df = pd.DataFrame(offset_rows)
display(offset_df)

assert offset_df.loc[offset_df.topic == "cpg-nodes", "messages"].iloc[0] > 0
assert offset_df.loc[offset_df.topic == "cpg-edges", "messages"].iloc[0] > 0
assert offset_df.loc[offset_df.topic == "cpg-metadata", "messages"].iloc[0] > 0
print("\n[OK] cpg-nodes / cpg-edges / cpg-metadata đều có message.")

offset_df.to_csv(OUTPUT_DIR / "task3_offsets.csv", index=False)
print(f"[OK] Saved -> {OUTPUT_DIR / 'task3_offsets.csv'}")


### 5. Đọc sample message từ Kafka (nodes / edges / metadata)

Chứng minh payload trên broker đúng schema đã thiết kế.


In [ ]:
def read_one(topic_name, timeout_ms=8000):
    c = KafkaConsumer(
        topic_name,
        bootstrap_servers=KAFKA_BOOTSTRAP,
        auto_offset_reset="earliest",
        consumer_timeout_ms=timeout_ms,
        value_deserializer=lambda b: json.loads(b.decode("utf-8")),
        key_deserializer=lambda b: b.decode("utf-8") if b else None,
    )
    for msg in c:
        c.close()
        return msg.key, msg.value
    c.close()
    return None, None

samples = {}
for topic in ["cpg-nodes", "cpg-edges", "cpg-metadata"]:
    key, value = read_one(topic)
    samples[topic] = {"key": key, "value": value}
    print(f"\n===== SAMPLE from {topic} =====")
    print("Kafka key:", key)
    print(json.dumps(value, indent=2, ensure_ascii=False) if value else "(no message)")
    if value:
        assert "schema_version" in value, f"{topic} thiếu schema_version"
        assert "event_time" in value, f"{topic} thiếu event_time"

sample_path = OUTPUT_DIR / "task3_kafka_samples.json"
sample_path.write_text(json.dumps(samples, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n[OK] Saved samples -> {sample_path}")


### 6. Tóm tắt bằng chứng Task 2 & Task 3 + Reflection


In [ ]:
print("╔══════════════════════════════════════════════════════════╗")
print("║        TASK 2 & 3 – EVIDENCE SUMMARY (Lab Report)        ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Parser script     : cpg_parser.py                       ║")
print(f"║  AST library       : Python ast                          ║")
print(f"║  Schema version    : {SCHEMA_VERSION:<39}║")
print(f"║  Demo files parsed : {DEMO_LIMIT:<39}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Kafka topics                                            ║")
for _, row in offset_df.iterrows():
    print(f"║    {row['topic']:<14} messages = {int(row['messages']):<28}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Artifacts in output/                                    ║")
print("║    task2_dryrun.log / task2_parse_stats.csv              ║")
print("║    task3_topics_list.log / task3_topics_describe.log     ║")
print("║    task3_emit.log / task3_offsets.csv                    ║")
print("║    task3_kafka_samples.json                              ║")
print("╚══════════════════════════════════════════════════════════╝")


### 7. Reflection – Task 3

**What worked**
- 4 topic tách rõ theo loại event → Neo4j chỉ cần subscribe nodes/edges, Spark/Mongo chỉ cần metadata.
- `schema_version` + `event_time` trên mọi message; Kafka key = stable id hỗ trợ sink idempotent ở Task 4.
- `advertised.listeners=PLAINTEXT://localhost:9092` giúp producer Windows nói chuyện được với broker trong WSL.

**What failed**
- Lần đầu broker advertise hostname WSL (`DESKTOP-....localdomain`) khiến client Windows kết nối metadata fail.

**How resolved**
- Sửa `server.properties` (`listeners=0.0.0.0:9092`, `advertised.listeners=localhost:9092`) và đóng gói vào `scripts/setup_kafka.sh`.

## **Task 4: Graph Topology Ingestion into Neo4j**

### Mục tiêu
- Kết nối **Neo4j Kafka Connector Sink** với hai topic `cpg-nodes` và `cpg-edges`.
- Ghi trực tiếp cấu trúc đồ thị từ **Kafka** vào **Neo4j**, không thông qua Spark.
- Sử dụng `MERGE` thay cho `CREATE` để đảm bảo **idempotent**, tức là khi xử lý lại dữ liệu sẽ không tạo bản ghi trùng.

### Kiến trúc

```text
cpg_parser.py  -->  Kafka (cpg-nodes, cpg-edges)  -->  Kafka Connect  -->  Neo4j
                    localhost:9092                      :8083               :7687
```

### Docker Compose

| Dịch vụ | Image | Cổng |
|:---|:---|:---:|
| ZooKeeper | confluentinc/cp-zookeeper:7.6.1 | 2181 |
| Kafka | confluentinc/cp-kafka:7.6.1 | 9092 |
| Kafka Connect | confluentinc/cp-kafka-connect:7.6.1 | 8083 |
| Neo4j | neo4j:5.20.0-community | 7474, 7687 |

### 0. Cấu hình đường dẫn

In [9]:

import subprocess as _sp
from pathlib import Path

# Chuyển đường dẫn Windows sang định dạng WSL
def to_wsl_path(p):
    p = str(p).replace("\\", "/")
    if len(p) >= 2 and p[1] == ":":
        return f"/mnt/{p[0].lower()}{p[2:]}"
    return p

# Tìm Python của Conda trong WSL
def find_conda_wsl():
    for name in ["miniconda3", "anaconda3", "miniforge3"]:
        path = f"~/{name}/bin/python3"
        r = _sp.run(
            ["wsl", "-e", "bash", "-c", f"ls {path}"],
            capture_output=True,
            text=True,
        )
        if r.returncode == 0:
            return r.stdout.strip()

    # Nếu không tìm thấy thì dùng Python mặc định
    return "python3"


WSL_ROOT = to_wsl_path(PROJECT_ROOT)
TASK4_WSL = f"{WSL_ROOT}/task4"
CONDA_WSL = find_conda_wsl()

print(f"WSL_ROOT  = {WSL_ROOT}")
print(f"TASK4_WSL = {TASK4_WSL}")
print(f"CONDA_WSL = {CONDA_WSL}")

def run_wsl(bash_cmd, timeout=180):
    """Chạy lệnh bash trong WSL, trả về CompletedProcess."""
    return subprocess.run(
        ["wsl", "-e", "bash", "-c", bash_cmd],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )

WSL_ROOT  = /mnt/e/BD
TASK4_WSL = /mnt/e/BD/task4
CONDA_WSL = /home/myha/miniconda3/bin/python3


### 1. Khởi động các dịch vụ của Task 4

In [10]:
def start_task4_services():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/start.py', timeout=400)
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

start_task4_services()

Kiểm tra và xóa container cũ (nếu có)...
  Đã xóa container: cpg-zookeeper (trạng thái trước đó: running)
  Đã xóa container: cpg-kafka (trạng thái trước đó: running)
  Đã xóa container: cpg-kafka-connect (trạng thái trước đó: running)
  Đã xóa container: cpg-neo4j (trạng thái trước đó: running)
Khởi động Docker Stack...
  Container cpg-zookeeper Started
  Container cpg-neo4j Started
  Container cpg-kafka Started
  Container cpg-kafka-connect Started
Chờ Kafka Broker sẵn sàng (15 giây)...
Tạo các Kafka topic...
  cpg-nodes: Created topic cpg-nodes.
  cpg-edges: Created topic cpg-edges.
  cpg-metadata: Created topic cpg-metadata.
  cpg-errors: Created topic cpg-errors.
Chờ Kafka Connect và plugin Neo4j khởi động (có thể mất 3–5 phút)...
  [1/36] Chờ 5 giây...
  [2/36] Chờ 5 giây...
  [3/36] Chờ 5 giây...
  [4/36] Chờ 5 giây...
  Đã tìm thấy plugin Neo4j: org.neo4j.connectors.kafka.sink.Neo4jConnector
Docker Stack và Kafka Connect đã sẵn sàng.



### 2. Đăng ký Neo4j Connector Sink

Sử dụng Kafka Connect REST API để đăng ký hai connector:

- `neo4j-cpg-nodes-sink`: đọc dữ liệu từ topic `cpg-nodes` và ghi các node vào Neo4j bằng câu lệnh `MERGE`.
- `neo4j-cpg-edges-sink`: đọc dữ liệu từ topic `cpg-edges` và ghi các relationship vào Neo4j bằng câu lệnh `MERGE`.

In [11]:
def register_neo4j_connectors():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/setup.py')
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

register_neo4j_connectors()

Đang chờ Kafka Connect sẵn sàng... OK

Kiểm tra plugin Neo4j:
  Tìm thấy: org.neo4j.connectors.kafka.sink.Neo4jConnector (phiên bản 5.1.10)
  Tìm thấy: org.neo4j.connectors.kafka.source.Neo4jConnector (phiên bản 5.1.10)

Đăng ký các connector...
  [neo4j-cpg-nodes-sink] Đăng ký thành công
  [neo4j-cpg-edges-sink] Đăng ký thành công

Chờ connector khởi động (30 giây)...

Trạng thái connector:
  neo4j-cpg-nodes-sink: RUNNING, tasks=['RUNNING']
  neo4j-cpg-edges-sink: RUNNING, tasks=['RUNNING']



### 3. Gửi CPG Events lên Kafka

Chạy `cpg_parser.py` với 5 tệp Python đầu tiên để phát (emit) các sự kiện lên các Kafka topic sau:

- `cpg-nodes`: chứa các node của Code Property Graph (AST, CFG, DFG).
- `cpg-edges`: chứa các cạnh giữa các node, bao gồm quan hệ AST (cha–con), CFG (luồng điều khiển), DFG (luồng dữ liệu) và `CALL`.
- `cpg-metadata`: chứa thông tin metadata của từng tệp Python được xử lý.

In [12]:
def emit_cpg_events():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/emit.py', timeout=300)
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

emit_cpg_events()

=== Starting CPG Parser Service ===
Discovered CSV : /mnt/e/BD/output/discovered_files.csv
Repo Root      : /mnt/e/BD/peft
Kafka Brokers  : localhost:9092
Schema Version : 1.0.0
Dry Run Mode   : False

Found 5 source files to parse.
[OK] Connected to Kafka Producer successfully.

[1/5] Processing: docs/source/_config.py ... Parsed and Sent (Nodes: 5, Edges: 6)
[2/5] Processing: examples/adamss_finetuning/glue_adamss_asa_example.py ... Parsed and Sent (Nodes: 2222, Edges: 3931)
[3/5] Processing: examples/adamss_finetuning/glue_adamss_asa_manual_example.py ... Parsed and Sent (Nodes: 2175, Edges: 3781)
[4/5] Processing: examples/adamss_finetuning/image_classification_adamss_asa.py ... Parsed and Sent (Nodes: 2319, Edges: 4123)
[5/5] Processing: examples/alora_finetuning/alora_finetuning.py ... Parsed and Sent (Nodes: 1227, Edges: 2045)

=== Parser Run Summary ===
Total Source Files processed : 5
Successfully Parsed          : 5
Failed                       : 0
Total Nodes emitted        

### 4. Kiểm chứng dữ liệu đã được đưa vào Neo4j

Sau khi các sự kiện được gửi lên Kafka, Kafka Connect sẽ tự động đọc dữ liệu từ các topic và ghi vào Neo4j.

Cell dưới đây sẽ chờ khoảng **90 giây** để Kafka Connect xử lý dữ liệu theo từng lô (batch), sau đó truy vấn Neo4j và hiển thị kết quả.

**Các nội dung cần kiểm tra:**

1. Số lượng `CpgNode` lớn hơn 0.
2. Số lượng `CPG_EDGE` lớn hơn 0, bao gồm các loại quan hệ: `AST`, `CFG`, `DFG` và `CALL`.
3. `duplicates = 0`, xác nhận câu lệnh `MERGE` hoạt động đúng và không tạo dữ liệu trùng lặp khi xử lý lại.

In [13]:
def verify_neo4j_ingestion():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/verify.py')
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

verify_neo4j_ingestion()

Tổng số CpgNode  : 7,200
Tổng số CPG_EDGE : 12,830

Thống kê loại node (Top 10):
  Load                         1,795
  Name                         1,496
  Constant                     765
  Attribute                    474
  Call                         455
  Store                        323
  keyword                      305
  Expr                         212
  Assign                       198
  Dict                         128

Thống kê loại relationship:
  AST                          7,195
  CFG                          4,748
  DFG                          517
  CALL                         370

Các tệp đã xử lý:
  examples/adamss_finetuning/image_classification_adamss_asa.py  ->  2,103 node
  examples/adamss_finetuning/glue_adamss_asa_example.py  ->  1,990 node
  examples/adamss_finetuning/glue_adamss_asa_manual_example.py  ->  1,954 node
  examples/alora_finetuning/alora_finetuning.py  ->  1,148 node
  docs/source/_config.py  ->  5 node

Node mẫu:
{
  "schema_version": "1.0.0",

### 5. Kết quả

**1. Triển khai hạ tầng**  
Kiểm tra trạng thái các container trong stack:

<div align="center">
  <img src="result/task4/1.png" alt="Docker status" width="800"/>
  <p><i>Hình 4.5.1: Các dịch vụ Docker (Zookeeper, Kafka, Connect, Neo4j) đang hoạt động (Up)</i></p>
</div>  

**2. Trạng thái Connectors**  
Xác nhận các sink connector đã được đăng ký và đang chạy:

<div align="center">
  <img src="result/task4/2.png" alt="Connector status" width="800"/>
  <p><i>Hình 4.5.2: Trạng thái RUNNING của neo4j-cpg-nodes-sink và neo4j-cpg-edges-sink</i></p>
</div>  

**3. Kiểm chứng dữ liệu trong Neo4j**  
Dưới đây là kết quả thống kê số lượng Nodes và Relationships đã được ingestion thành công:

<div align="center">
  <img src="result/task4/3.1.png" alt="Neo4j Nodes" width="800"/>
  <p><i>Hình 4.5.3: Kết quả truy vấn đếm CpgNode (Tổng: 7,200)</i></p>
</div>

<div align="center">
  <img src="result/task4/3.2.png" alt="Neo4j Edges" width="800"/>
  <p><i>Hình 4.5.4: Kết quả truy vấn đếm CPG_EDGE (Tổng: 12,830)</i></p>
</div>

### 6. Reflection

**What worked**
- Docker Compose giúp khởi động toàn bộ stack (ZooKeeper, Kafka, Kafka Connect và Neo4j) chỉ với một lệnh, giúp việc thiết lập môi trường nhanh và thuận tiện.
- Neo4j Kafka Connector Sink tự động đọc dữ liệu từ các topic `cpg-nodes` và `cpg-edges`, sau đó thực thi câu lệnh Cypher để ghi trực tiếp vào Neo4j mà không cần thông qua Spark.
- Sử dụng `MERGE` thay cho `CREATE` đảm bảo tính idempotent. Sau khi emit dữ liệu nhiều lần, số lượng node và relationship không thay đổi, đồng thời `duplicates = 0`.

**What failed**
- Neo4j Connector phiên bản 5.x thay đổi khóa cấu hình từ `neo4j.topic.cypher.<name>` sang `neo4j.cypher.topic.<name>`, khiến connector không hoạt động khi sử dụng cấu hình của phiên bản cũ.
- Connector ghi `cpg-edges` chỉ hoạt động khi các node tương ứng đã tồn tại trong Neo4j. Nếu node connector chưa xử lý xong, các relationship sẽ không được tạo.
- Đường dẫn tệp sinh trên Windows sử dụng dấu `\`, trong khi các script chạy trên WSL yêu cầu định dạng `/`, dẫn đến lỗi không tìm thấy tệp.

**How resolved**
- Cập nhật cấu hình connector theo đúng cú pháp của Neo4j Connector 5.x và kiểm tra plugin thông qua Kafka Connect REST API trước khi đăng ký connector.
- Khởi động Kafka Connect trước, chờ connector sẵn sàng rồi mới emit dữ liệu để đảm bảo các node được ghi vào Neo4j trước khi xử lý relationship.
- Chuyển đổi đường dẫn từ định dạng Windows sang định dạng WSL trước khi truyền vào các script, giúp các bước xử lý hoạt động ổn định trên cả hai môi trường.